In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")
sys.path.append("../data/embeddings")
sys.path.append("../data/biasbios")
sys.path.append("../data/embeddings/biasbios")
sys.path.append("../nullitoutsrc")
import classifier
import debias
import gensim
import codecs
import json
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from gensim.models.keyedvectors import Word2VecKeyedVectors
from gensim.models import KeyedVectors
import numpy as np
import random
import sklearn
from sklearn import model_selection
from sklearn import cluster
from sklearn import metrics
from sklearn.manifold import TSNE
from sklearn.svm import LinearSVC, SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import Pipeline
from pytorch_transformers import BertTokenizer, BertModel, BertForMaskedLM

import scipy
from scipy import linalg
from scipy import sparse
from scipy.stats.stats import pearsonr
import tqdm
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier, SGDRegressor, Perceptron, LogisticRegression
from sklearn.utils import shuffle

%matplotlib inline
matplotlib.rcParams['agg.path.chunksize'] = 10000

import warnings
warnings.filterwarnings("ignore")

import pickle
from collections import defaultdict, Counter
from typing import List, Dict

import torch
from torch import utils

import pytorch_lightning as pl
from pytorch_lightning import Trainer
import copy
import pandas as pd
from gensim.models import FastText
import time
from gensim.scripts.glove2word2vec import glove2word2vec

STOPWORDS = set(["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"])

In [ ]:


def load_dataset(path):
    
    with open(path, "rb") as f:
        
        data = pickle.load(f)
    return data

def load_dictionary(path):
    
    with open(path, "r", encoding = "utf-8") as f:
        
        lines = f.readlines()
        
    k2v, v2k = {}, {}
    for line in lines:
        
        k,v = line.strip().split("\t")
        v = int(v)
        k2v[k] = v
        v2k[v] = k
    
    return k2v, v2k
    
def count_profs_and_gender(data: List[dict]):
    
    counter = defaultdict(Counter)
    for entry in data:
        gender, prof = entry["g"], entry["p"]
        counter[prof][gender] += 1
        
    return counter


In [ ]:
train = load_dataset("../data/biasbios/train.pickle")
dev = load_dataset("../data/biasbios/dev.pickle")
test = load_dataset("../data/biasbios/test.pickle")

p2i, i2p = load_dictionary("../data/biasbios/profession2index.txt")
g2i, i2g = load_dictionary("../data/biasbios/gender2index.txt")
counter = count_profs_and_gender(train+dev+test)

In [ ]:
train[0]

In [ ]:
print(p2i)

In [ ]:
counter = count_profs_and_gender(train+dev+test)
f,m = 0., 0.
prof2fem = dict()

for k, values in counter.items():
    f += values['f']
    m += values['m']
    prof2fem[k] = values['f']/(values['f'] + values['m'])

print(f / (f + m))
print(prof2fem)

In [ ]:
print(p2i)

In [ ]:
import nltk

def nltk_tokenization(text):
    tokens = nltk.word_tokenize(text)
    return tokens

def built_in_tokenization(text):
    tokens = text.split()
    return tokens

In [ ]:
x_train = [x['hard_text'] for x in train]
y_train = [x['p'] for x in train]

x_dev = [x['hard_text'] for x in dev]
y_dev = [x['p'] for x in dev]

x_test = [x['hard_text'] for x in test]
y_test = [x['p'] for x in test]

clf_vectorizer = Pipeline([
    ('vectorizer', CountVectorizer(tokenizer=built_in_tokenization)),
    ('selection', SelectKBest(chi2, k=10000)),
    # ('classifier', LogisticRegression())
    # ('classifier', SGDClassifier(warm_start=True, loss='log', n_jobs=64, max_iter=75, random_state=0))
])
clf_vectorizer.fit(x_train, y_train)
# clf.score(x_dev, y_dev)

### Get BOW representatons 

In [ ]:
x_train_one_hot = clf_vectorizer.named_steps['selection'].transform(clf_vectorizer.named_steps['vectorizer'].transform(x_train))
x_dev_one_hot = clf_vectorizer.named_steps['selection'].transform(clf_vectorizer.named_steps['vectorizer'].transform(x_dev))
x_test_one_hot = clf_vectorizer.named_steps['selection'].transform(clf_vectorizer.named_steps['vectorizer'].transform(x_test))

## Classify by profession

In [ ]:
#clf = LinearSVC(max_iter = 10) #LogisticRegression()
#clf = SGDClassifier(warm_start = True, loss = 'log', n_jobs = 64, max_iter = 250, random_state = 0)

clf = LogisticRegression(warm_start = True, penalty = 'l2',
                         solver = "sag", multi_class = 'multinomial', fit_intercept = True,
                         verbose = 10, max_iter = 4, n_jobs = 64, random_state = 1)
#clf = SGDClassifier(loss= 'log', max_iter= 3000, fit_intercept= True, class_weight= None, n_jobs= 100)
#clf = LinearSVC(max_iter = 50) #LogisticRegression()

start = time.time()
# idx = np.random.rand(X_train.shape[0]) < 1.0
# clf.fit(X_train[idx], Y_train[idx])
clf.fit(x_train_one_hot, y_train)
print("time: {}".format(time.time() - start))
print(clf.score(x_dev_one_hot, y_dev))
#print(clf.score(X_test, Y_test))
clf_original = copy.deepcopy(clf)

In [ ]:
print(clf.score(x_test_one_hot, y_test))


#### Confusion Matrix

### perform debiasing

In [ ]:
def get_projection_matrix(num_clfs, X_train, Y_train, X_dev, Y_dev, Y_train_task, Y_dev_task, dim, all_data_prob, by_class = False):

    is_autoregressive = True
    min_acc = 0.
    dim = 300
    n = num_clfs
    random_subset = 1
    start = time.time()
    TYPE= "logistic"
    penalty = "l2"
    MLP = False
    
    if MLP:
        x_train_gender = np.matmul(X_train, clf.coefs_[0]) + clf.intercepts_[0]
        x_dev_gender = np.matmul(X_dev, clf.coefs_[0]) + clf.intercepts_[0]
    else:
        x_train_gender = X_train.copy()
        x_dev_gender = X_dev.copy()
        
    
    if TYPE == "sgd":
        print("using sgd")
        gender_clf = SGDClassifier
        params = {'alpha': 0.01, 'penalty': penalty, 'loss': 'hinge', 'fit_intercept': True, 'class_weight': "balanced", 'n_jobs': 16}
    elif TYPE == "svm":
        gender_clf = LinearSVC
        params = {'fit_intercept': True, 'C': 0.3, 'class_weight': None, "dual": False}
    elif TYPE == "perceptron":
        gender_clf = Perceptron
        params = {'max_iter': 1000, 'fit_intercept': True, 'class_weight': None}
    elif TYPE == "logistic":
        gender_clf = LogisticRegression
        params = {}
        
    result, r, Ws, accuracy_scores = debias.get_debiasing_projection(gender_clf, params, n, dim, is_autoregressive, min_acc,
                                              x_train_gender, Y_train, x_dev_gender, Y_dev,
                                       Y_train_main=Y_train_task, Y_dev_main=Y_dev_task, 
                                        by_class = by_class)
    print("time: {}".format(time.time() - start))
    return result

# was c=0.15, num_clfs=130
num_clfs = 4
Y_dev_gender = np.array([g2i[d["g"]] for d in dev])
Y_test_gender = np.array([g2i[d["g"]] for d in test])
Y_train_gender = np.array([g2i[d["g"]] for d in train])
P, rowspace_projs, Ws = get_projection_matrix(num_clfs, x_train_one_hot, Y_train_gender, x_dev_one_hot, Y_dev_gender, np.array(y_train), y_dev, 300, 0.0, by_class= True)



In [ ]:
from importlib import reload

In [ ]:
reload(debias)

In [ ]:
with open("P.bios.fasttext.{}.svm.pickle".format(num_clfs), "wb") as f:
    
    pickle.dump(P, f)
    
np.linalg.norm(Ws[-1])

In [ ]:
Ws[-50].dot(Ws[0].T)

In [ ]:
gender_clf = LogisticRegression(warm_start = True, penalty = 'l2',
                         solver = "sag", multi_class = 'multinomial', fit_intercept = True,
                         verbose = 10, max_iter = 7, n_jobs = 64, random_state = 1)

gender_clf.fit(P.dot(X_train.T).T, Y_train_gender)
gender_clf.score(P.dot(X_dev.T).T, Y_dev_gender)

In [ ]:
gender_clf.score(P.dot(X_train.T).T, Y_train_gender)

### Perform finetuning

In [ ]:

clf = LogisticRegression(warm_start = True, penalty = 'l2',
                         solver = "sag", multi_class = 'multinomial', fit_intercept = True,
                         verbose = 10, max_iter = 6, n_jobs = 64, random_state = 0, class_weight = None)
# 5
#clf = SGDClassifier(verbose=10, n_jobs = 32)
clf.fit(P.dot(X_train.T).T, Y_train)
#clf.fit(X_train, Y_train)


In [ ]:
print(clf.score(P.dot(X_test.T).T, Y_test))

#### test again

In [ ]:
def get_TPR(y_pred, y_true, p2i, i2p, gender):
    
    scores = defaultdict(Counter)
    prof_count_total = defaultdict(Counter)
    
    assert len(y_pred) == len(y_true) == len(gender)
               
    for y_hat, y, g in zip(y_pred, y_true, gender):
                
        if y == y_hat:
            
            scores[i2p[y]][g] += 1
        
        prof_count_total[i2p[y]][g] += 1
    
    tprs = defaultdict(dict)
    tprs_change = dict()
    tprs_ratio = []
    
    for profession, scores_dict in scores.items():
        
        good_m, good_f = scores_dict["m"], scores_dict["f"]
        prof_total_f = prof_count_total[profession]["f"]
        prof_total_m = prof_count_total[profession]["m"]
        tpr_m = (good_m) / prof_total_m
        tpr_f = (good_f) / prof_total_f
        
        tprs[profession]["m"] = tpr_m
        tprs[profession]["f"] = tpr_f
        tprs_ratio.append(0)
        tprs_change[profession] = tpr_f - tpr_m
        
    return tprs, tprs_change, np.mean(np.abs(tprs_ratio))

def get_FPR(y_pred, y_true, p2i, i2p, gender):
    
    fp = defaultdict(Counter)
    neg_count_total = defaultdict(Counter)
    pos_count_total = defaultdict(Counter)
    
    
               
    for y_hat, y, g in zip(y_pred, y_true, gender):
        
        if y != y_hat:
            
            fp[i2p[y_hat]][g] += 1 # count false positives for y_hat
        
        neg_count_total[i2p[y_hat]][g] += 1 # count "not y_hat"
                
    fprs = defaultdict(dict)
    fprs_change = dict()
    fprs_ratio = []
    
    #print(prof_count_total)
    
    for profession, scores_dict in fp.items():
        
        good_m, good_f = scores_dict["m"], scores_dict["f"]
        prof_total_f = neg_count_total[profession]["f"]
        prof_total_m = neg_count_total[profession]["m"]
        print(profession)
        fpr_m = (good_m+good_f) / prof_total_m
        fpr_f = (good_m+good_f) / prof_total_f
        
        fprs[profession]["m"] = fpr_m
        fprs[profession]["f"] = fpr_f
        fprs_ratio.append(0)
        fprs_change[profession] = fpr_m - fpr_f
        
    return fprs, fprs_change, np.mean(np.abs(fprs_ratio))    
    
def similarity_vs_tpr(tprs, word2vec, title, measure, prof2fem):
    
    professions = list(tprs.keys())
    #
    """ 
    sims = dict()
    gender_direction = word2vec["he"] - word2vec["she"]
    
    for p in professions:
        sim = word2vec.cosine_similarities(word2vec[p], [gender_direction])[0]
        sims[p] = sim
    """
    tpr_lst = [tprs[p] for p in professions]
    sim_lst = [prof2fem[p] for p in professions]

    #professions = [p.replace("_", " ") for p in professions if p in word2vec]
    
    plt.plot(sim_lst, tpr_lst, marker = "o", linestyle = "none")
    plt.xlabel("% women", fontsize = 13)
    plt.ylabel(r'$GAP_{female,y}^{TPR}$', fontsize = 13)
    for p in professions:
        x,y = prof2fem[p], tprs[p]
        plt.annotate(p , (x,y), size = 7, color = "red")
    
    plt.ylim(-0.4,0.55)
    z = np.polyfit(sim_lst, tpr_lst, 1)
    p = np.poly1d(z)
    plt.plot(sim_lst,p(sim_lst),"b--")

    plt.savefig("{}_vs_bias_{}_fasttext".format(measure, title), dpi = 600)
    print("Correlation: {}; p-value: {}".format(*pearsonr(sim_lst, tpr_lst)))
    plt.show()
    
def rms_diff(tpr_diff):
    
    return np.sqrt(np.mean(tpr_diff**2))
    
def save_vecs_and_words(vecs, words):
    def to_string(arr):
        return "\t".join([str(x) for x in arr])
    
    with open("vecs.txt", "w") as f:
        for v in vecs:
            assert len(v) == 300
            f.write(to_string(v) + "\n")
    
    with open("labels.txt", "w") as f:
            f.write("Profession\n")
            for w in words:
                f.write(w + "\n")

In [ ]:


y_pred_before = clf_original.predict(X_test)
test_gender = [d["g"] for d in test]
dev_gender = [d["g"] for d in dev]
train_gender = [d["g"] for d in train]

tprs_before, tprs_change_before, mean_ratio_before = get_TPR(y_pred_before, Y_test, p2i, i2p, test_gender)
similarity_vs_tpr(tprs_change_before, word2vec, "before", "TPR", prof2fem)


#y_pred_after = clf.predict(X_test.dot(P))
y_pred_after = clf.predict(X_test.dot(P))
tprs, tprs_change_after, mean_ratio_after = get_TPR(y_pred_after, Y_test, p2i, i2p, test_gender)
similarity_vs_tpr(tprs_change_after, word2vec, "after", "TPR", prof2fem)

change_vals_before = np.array(list((tprs_change_before.values())))
change_vals_after = np.array(list(tprs_change_after.values()))

print("rms-diff before: {}; rms-diff after: {}".format(rms_diff(change_vals_before), rms_diff(change_vals_after)))

In [ ]:
def tsne_by_gender(vecs, labels, title, words = None):

  tsne = TSNE(n_components=2, random_state=0)
  vecs_2d = tsne.fit_transform(vecs)
  num_labels = len(set(labels.tolist()))

  names = ["class {}".format(i) for i in range(num_labels)]
  plt.figure(figsize=(6, 5))
  colors = 'r', 'b', 'orange'
  for i, c, label in zip(set(labels.tolist()), colors, names):
    print(len(vecs_2d[labels == i, 0]))
    plt.scatter(vecs_2d[labels == i, 0], vecs_2d[labels == i, 1], c=c,
                label=label, alpha = 0.6)
  plt.legend(loc = "upper right")
  plt.title(title)
  
  if words is not None:
        k = 60
        for i in range(k):
            
            j = np.random.choice(range(len(words)))
            label = labels[i]
            w = words[j]
            x,y = vecs_2d[i]
            plt.annotate(w , (x,y), size = 10, color = "black" if label == 1 else "black")
            
  plt.show()
  return vecs_2d

In [ ]:
prof = "professor"
idx = np.random.rand(X_dev.shape[0]) < 0.1
prof_idx = Y_dev == p2i[prof]
n = 800
tsne_by_gender(X_dev[prof_idx][:n], np.array(dev_gender)[prof_idx][:n], "tsne by gender, before, {}".format(prof))
tsne_by_gender((X_dev[prof_idx].dot(P))[:n], np.array(dev_gender)[prof_idx][:n], "tsne by gender, after. {}".format(prof))

In [ ]:
def count_profs(data_y, i2p):
    d = Counter()
    for y in data_y:
        d[i2p[y]] += 1
    return d

In [ ]:
count_profs(Y_train, i2p)

In [ ]:
count_profs(Y_dev, i2p)

In [ ]:
p0 = all_Ps[0]
p1 = all_Ps[1]
p2 = all_Ps[2]

In [ ]:
p2.dot(p2.T)<

In [ ]:
p0 = all_Ps[0]
p1 = all_Ps[1]
p2 = all_Ps[2]

p2.dot(p2.T)

In [ ]:
def get_common_words(data: List[dict], word2vec_model):
    
    words_counter = Counter()
    vecs = []
    all_words = []
    
    
    for entry in tqdm.tqdm(data, total = len(data)):
        
        y = p2i[entry["p"]]
        words = entry["hard_text"].split(" ")
        all_words.extend(words)
            
    
    words_counter = Counter(all_words)
    common_words = [w for w in words_counter if words_counter[w] > 10 and (w in word2vec_model)]
    common_vecs = [word2vec_model[w] for w in common_words]
    
    return common_words, common_vecs
        
        

In [ ]:
words, vecs = get_common_words(train,word2vec)

vecs_normed = vecs / np.linalg.norm(vecs, keepdims = True)
ws_normed = [w/np.linalg.norm(w) for w in all_ws]
k = 1000
groups, labels = [], []

for i,w in enumerate(ws_normed):
    print("INLP ITERATION: {}".format(i))
    sims_i = w.dot(vecs_normed.T).squeeze(0)
    zipped = zip(words, vecs_normed, sims_i)
    zipped = sorted(list(zipped), key = lambda tuple: -abs(tuple[2]))
    ws,vs, sims = list(zip(*zipped))
    print(ws[:k])
    groups.append(vs[:k])
    labels.append(i)
    print("------------------------------------------------------------------")

In [ ]:
def visualize_gender_subspace(vecs, labels):
    
        N = len(labels)
        
        all_vecs = []
        all_labels = []
        
        for vecs_list,l in zip(vecs, labels):
            
            all_labels.append(np.ones(len(vecs_list)) * l)
        
        all_vecs_np = np.concatenate(vecs, axis = 0)
        all_labels_np = np.concatenate(all_labels, axis = 0)
        tsne = TSNE(n_components=2, random_state=0)
        vecs_2d = tsne.fit_transform(all_vecs_np)
    
        fig, ax = plt.subplots()
        # define the colormap
        cmap = plt.cm.jet
        # extract all colors from the .jet map
        cmaplist = [cmap(i) for i in range(cmap.N)]
        # create the new map
        cmap = cmap.from_list('Custom cmap', cmaplist, cmap.N)
        # define the bins and normalize
        bounds = np.linspace(0, N, N + 1)
        norm = matplotlib.colors.BoundaryNorm(bounds, cmap.N)
        print("here")
        print(all_labels_np.shape)
        scat = ax.scatter(vecs_2d[:,0], vecs_2d[:,1], c=all_labels_np, cmap=cmap, norm=norm, alpha=0.15)
        cb = plt.colorbar(scat, spacing='proportional')#, ticks=bounds)
        cb.set_label("INLP iteration number")
        plt.savefig("INLP progress", dpi = 600)
        plt.show()

In [ ]:
#visualize_gender_subspace(groups, labels)

In [ ]:
word2vec

In [ ]:
i = 0
for w in Ws[:150]:
    
    #sims = word2vec.similar_by_vector(-w.squeeze(), topn = 60, restrict_vocab=None)
    most_similar_f, _ =  list(zip(*word2vec_bios.similar_by_vector(w.squeeze(), topn = 20, restrict_vocab=None)))
    most_similar_m, _ =  list(zip(*word2vec_bios.similar_by_vector(-w.squeeze(), topn = 20, restrict_vocab=None)))
    print("gender direction {}.\n most_similar male: {}\n; most_similar female: {}".format(i, most_similar_m, most_similar_f))
    print("=====================================")
    i += 1

In [ ]:
nurse_train = [r for r in train if r["p"] == "nurse"]
nurse_train[0]

In [ ]:
profs_train_str = np.array([i2p[y] for y in Y_train[:]])

In [ ]:
X_train.shape, profs_train_str.shape

In [ ]:
for p in p2i.keys():
    x_train_nurse = X_train[profs_train_str == p]
    y_train_nurse = Y_train[profs_train_str == p]
    print(p,len(x_train_nurse)/len(X_train)*100,"%", clf_original.score(x_train_nurse, y_train_nurse), clf.score(P.dot(x_train_nurse.T).T, y_train_nurse))

In [ ]:
x_train_nurse.shape, y_train_nurse.shape

In [ ]:
{'professor': 0.45118956904580476, 'chiropractor': 0.26558891454965355, 'psychologist': 0.6223011751844766, 'architect': 0.23712053792148718, 'physician': 0.507688318423441, 'nurse': 0.9085446207369142, 'dentist': 0.35589474411216243, 'surgeon': 0.14857228961048746, 'rapper': 0.09665955934612651, 'model': 0.8283124500133298, 'photographer': 0.35721920736720936, 'composer': 0.16392857142857142, 'comedian': 0.21150410861021793, 'filmmaker': 0.3295762590954487, 'paralegal': 0.8483305036785512, 'journalist': 0.49488721804511276, 'personal_trainer': 0.45670391061452514, 'teacher': 0.603111879476414, 'painter': 0.4579886246122027, 'attorney': 0.38316925813475633, 'accountant': 0.36818825194621374, 'software_engineer': 0.1576889661164205, 'poet': 0.49080017115960634, 'dj': 0.1420875420875421, 'pastor': 0.24052132701421802, 'yoga_teacher': 0.8454600120264583, 'dietitian': 0.9273504273504274, 'interior_designer': 0.8086124401913876}